# Chaos: the Lorenz attractor

In 1963 Edward Lorenz reduced a model of a convecting fluid layer to three ordinary
differential equations, restarted a run from printed output rounded to three decimals
instead of the six the machine held, and watched the new run diverge from the old one
until they had nothing in common.

That accident is the reason weather forecasts stop, why forecast centres run *ensembles*
rather than single trajectories, and why a climate projection is a claim about a
distribution rather than about a particular Tuesday. This notebook builds the picture and
measures the divergence.

Unlike notebooks 045 and 047, there is no data to download. Everything here comes out of
three equations and an integrator you can read in full.

## Learning objectives

By the end, you can:

- integrate a system of ODEs with a scheme you can write down, and check its order;
- explain what a strange attractor is and why the trajectory never repeats;
- quantify sensitive dependence with a Lyapunov exponent instead of hand-waving; and
- say what an ensemble forecast is for, and when it stops being informative.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import plotly.graph_objects as go

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "README.md").exists(), "Open the course project folder first."

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from climate_course.lorenz import (
    CITATION,
    CLASSIC_BETA,
    CLASSIC_RHO,
    CLASSIC_SIGMA,
    REFERENCE_LYAPUNOV,
    fixed_points,
    integrate,
    largest_lyapunov,
    lorenz_derivative,
    perturbed_pair,
    separation,
)

print(f"sigma = {CLASSIC_SIGMA}, rho = {CLASSIC_RHO}, beta = {CLASSIC_BETA:.4f}")

## 1. The equations

$$\frac{dx}{dt} = \sigma (y - x), \qquad
  \frac{dy}{dt} = x(\rho - z) - y, \qquad
  \frac{dz}{dt} = xy - \beta z$$

`x` measures how fast the fluid is overturning, `y` the horizontal temperature contrast
between rising and sinking fluid, and `z` how far the vertical temperature profile has
been bent away from a straight line. They are dimensionless: this is a caricature of
convection, not a weather model, and nothing here forecasts anything.

Only two terms are non-linear — `xz` and `xy` — and that is the entire source of what
follows.

## 2. Where could the system sit still?

A steady state is a point where all three derivatives vanish. Before integrating anything,
find them: the answer explains the shape you are about to see.

In [ ]:
points = fixed_points()
print("steady states:")
for point in points:
    residual = np.abs(lorenz_derivative(point)).max()
    print(f"  ({point[0]:+8.4f}, {point[1]:+8.4f}, {point[2]:+8.4f})"
          f"   |largest derivative| = {residual:.2e}")

assert np.allclose(lorenz_derivative(points), 0.0, atol=1e-12)
print("\nAll three are genuine steady states.")

Two of these sit at the centre of the wings you are about to see. The trajectory circles
them forever and never settles onto either, because for these parameter values they are
*unstable*: a state nudged away from one is pushed further away rather than pulled back.

- The origin corresponds to no convection at all. Why is it a solution of the equations? **TODO**
- Predict: with two unstable steady states, does the trajectory (a) spiral into one,
  (b) settle into a repeating loop, or (c) neither? **TODO**

## 3. Integrate

The integrator is fourth-order Runge-Kutta, in `src/climate_course/lorenz.py`. Open it and
find the four derivative evaluations per step and the weights `1, 2, 2, 1`.

Euler's method — one evaluation, step along the tangent — is simpler and drifts off the
attractor visibly at this step size. `tests/test_lorenz.py` checks the order of accuracy
by halving the step and confirming the error falls by roughly a factor of sixteen.

In [ ]:
dt, steps = 0.005, 12000
trajectory = integrate([1.0, 1.0, 1.0], dt=dt, steps=steps)
settled = trajectory[2000:]        # discard the approach to the attractor

print("shape:", trajectory.shape, " (steps + 1, 3)")
print(f"time simulated: {dt * steps:.0f} dimensionless units")
print(f"x range: {settled[:, 0].min():+.1f} to {settled[:, 0].max():+.1f}")
print(f"z range: {settled[:, 2].min():+.1f} to {settled[:, 2].max():+.1f}")

# The attractor is bounded: the trajectory wanders forever inside a finite box.
assert np.abs(settled).max() < 100
assert np.isfinite(settled).all()
print("\nBounded and finite: the trajectory never escapes, and never repeats.")

## 4. The attractor

In [ ]:
time = np.arange(settled.shape[0]) * dt

butterfly = go.Figure(go.Scatter3d(
    x=settled[:, 0], y=settled[:, 1], z=settled[:, 2],
    mode="lines",
    line=dict(color=time, colorscale="Viridis", width=2,
              colorbar=dict(title="Time")),
    hovertemplate="x %{x:.2f}<br>y %{y:.2f}<br>z %{z:.2f}<extra></extra>",
))
butterfly.add_trace(go.Scatter3d(
    x=points[:, 0], y=points[:, 1], z=points[:, 2],
    mode="markers", marker=dict(size=5, color="black"),
    name="steady states", hoverinfo="skip",
))
butterfly.update_layout(
    title="The Lorenz attractor",
    height=680, margin=dict(l=0, r=0, t=48, b=0), showlegend=False,
    scene=dict(
        xaxis=dict(title="x  (overturning)"),
        yaxis=dict(title="y  (temperature contrast)"),
        zaxis=dict(title="z  (profile distortion)"),
        aspectmode="cube", camera=dict(eye=dict(x=1.5, y=1.5, z=0.6)),
    ),
)
butterfly

Rotate it. The trajectory is a single curve that never crosses itself and never closes.

- The two black dots are the unstable steady states. What is the trajectory doing
  relative to them? **TODO**
- Count wing switches by eye over a few seconds of the path. Is there a pattern? **TODO**
- The curve stays on a surface of essentially zero volume, yet never repeats. Explain to
  your partner how both can be true. **TODO**

## 5. Sensitive dependence, measured

Now Lorenz's accident. Take two starting states differing by `1e-9` in `x` alone — far
below any measurement precision — and integrate both.

In [ ]:
original, perturbed = perturbed_pair([1.0, 1.0, 1.0], offset=1e-9, dt=dt, steps=steps)
gap = separation(original, perturbed)
time_axis = np.arange(gap.size) * dt

for moment in (0.0, 5.0, 10.0, 15.0, 20.0, 25.0, 30.0, 40.0, 60.0):
    index = min(int(moment / dt), gap.size - 1)
    print(f"t = {moment:5.1f}   separation = {gap[index]:.3e}")

The gap sits at `1e-9` for a while, then grows by nine orders of magnitude, then stops
growing. All three phases matter.

In [ ]:
growth = go.Figure()
growth.add_trace(go.Scatter(x=time_axis, y=gap, mode="lines",
                            line=dict(color="#1f77b4", width=1.5), name="separation"))
growth.add_hline(y=1e-9, line=dict(color="gray", dash="dot"),
                 annotation_text="initial difference")
growth.add_hline(y=float(np.ptp(settled[:, 0])), line=dict(color="crimson", dash="dot"),
                 annotation_text="width of the attractor")
growth.update_layout(
    title="Separation between two runs started 1e-9 apart",
    xaxis_title="Time (dimensionless)",
    yaxis_title="Separation",
    yaxis_type="log", height=430, margin=dict(l=0, r=0, t=48, b=0),
    showlegend=False,
)
growth

Read the three phases off the figure:

1. **Flat.** The difference is too small to matter yet. **Until roughly t = TODO.**
2. **Straight, on a log axis.** Straight on a log axis means exponential growth. The slope
   is the thing we are about to measure.
3. **Flat again, at the width of the attractor.** Once the two states are on opposite
   wings they are as different as two states can be. The separation stops growing not
   because the system calmed down but because it ran out of room.

Phase 3 is the one people misread. Saturation is not predictability returning — it is
the point where the forecast has no information left.

## 6. Putting a number on it

The growth rate in phase 2 is the **largest Lyapunov exponent**, $\lambda$. A separation
grows like $e^{\lambda t}$, so the time for an error to double is $\ln 2 / \lambda$.

Measuring it from the curve above would only work inside phase 2. The standard method
instead lets the gap grow for a short interval, records the growth, then shrinks the gap
back along the same direction and repeats — so it never reaches saturation. Read
`largest_lyapunov` in `src/climate_course/lorenz.py` and find the renormalisation step.

In [ ]:
exponent = largest_lyapunov(renormalisations=2000, discard=200)
doubling_time = np.log(2) / exponent

print(f"largest Lyapunov exponent: {exponent:.3f} per time unit")
print(f"published value:           {REFERENCE_LYAPUNOV:.3f}")
print(f"an error doubles every     {doubling_time:.2f} time units")

assert exponent > 0, "A positive exponent is what makes this system chaotic."
print(f"\nStarting from 1e-9, reaching a separation of 10 takes about "
      f"{np.log(10 / 1e-9) / exponent:.0f} time units.")

Compare that last number with where phase 2 ends in the figure above. **TODO**

Now the forecasting consequence. Suppose you improve your observing network so that
initial errors shrink by a factor of 1000 — an enormous, expensive improvement.

- How much *extra* forecast lead time does that buy? **TODO**
- What does that imply about the return on ever-better observations? **TODO**

This is the argument for ensembles in one line: if you cannot remove the error, describe
what it does.

## 7. An ensemble

Instead of one trajectory, start a small cloud of them within `1e-3` of each other and
advance them together. This is a forecast ensemble in miniature.

In [ ]:
generator = np.random.default_rng(1963)
members = 300
start_point = integrate([1.0, 1.0, 1.0], dt=dt, steps=2000)[-1]
cloud = start_point + 1e-3 * generator.standard_normal((members, 3))

# 15 time units is enough: the cloud has covered the attractor well before then, and
# a longer run just animates a saturated blob.
ensemble_steps = 3000
paths = integrate(cloud, dt=dt, steps=ensemble_steps)   # (steps + 1, members, 3)
print("ensemble shape:", paths.shape)

spread = np.linalg.norm(paths - paths.mean(axis=1, keepdims=True), axis=-1).mean(axis=1)
for moment in (0.0, 2.0, 4.0, 6.0, 8.0, 10.0, 12.0, 15.0):
    index = min(int(moment / dt), spread.size - 1)
    print(f"t = {moment:5.1f}   mean spread = {spread[index]:8.3f}")

In [ ]:
shading = cloud[:, 0] - cloud[:, 0].mean()
limit = float(np.abs(shading).max())
frame_indices = range(0, paths.shape[0], 25)

def cloud_at(index):
    return go.Scatter3d(
        x=paths[index, :, 0], y=paths[index, :, 1], z=paths[index, :, 2],
        mode="markers",
        marker=dict(size=2.5, color=shading, colorscale="RdBu",
                    cmin=-limit, cmax=limit,
                    colorbar=dict(title="Start<br>offset in x")),
        hoverinfo="skip",
    )

frames = [go.Frame(data=[cloud_at(k)], name=f"{k * dt:.1f}") for k in frame_indices]
ensemble = go.Figure(data=[cloud_at(0)], frames=frames)
ensemble.update_layout(
    title=f"{members} states starting within 1e-3 of each other",
    height=680, margin=dict(l=0, r=0, t=48, b=0),
    scene=dict(
        xaxis=dict(title="x"), yaxis=dict(title="y"), zaxis=dict(title="z"),
        aspectmode="cube", camera=dict(eye=dict(x=1.5, y=1.5, z=0.6)),
    ),
    updatemenus=[dict(type="buttons", showactive=False, x=0.02, y=0.05, xanchor="left",
        buttons=[
            dict(label="Play", method="animate",
                 args=[None, dict(frame=dict(duration=70, redraw=True), mode="immediate")]),
            dict(label="Pause", method="animate",
                 args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")]),
        ])],
    sliders=[dict(active=0, x=0.14, len=0.82, y=0.05, currentvalue=dict(prefix="t = "),
        steps=[dict(label=f.name, method="animate",
                    args=[[f.name], dict(frame=dict(duration=0, redraw=True),
                                         mode="immediate")]) for f in frames])],
)
ensemble

Play it. The cloud stays a blob, then stretches into a filament, then wraps onto both
wings.

- At roughly what time does the cloud first occupy both wings? **TODO**
- The spread column above is not monotonic: it dips as well as grows. What is the cloud
  doing when the spread *falls*? **TODO**
- Once it does, what is the *mean* of the ensemble a good description of? Locate the
  ensemble mean in the figure and say whether any member is near it. **TODO**
- A forecaster says "the ensemble mean is the best forecast." When is that defensible
  here, and when is it actively misleading? **TODO**

The colouring is by each member's starting offset in `x`. Watch how neighbours stay
neighbours for a long time, and then do not — that is stretching and folding, and it is
what makes the attractor's fine structure.

## 8. Change the physics

`rho` is the Rayleigh number: how hard the layer is being heated. Everything so far used
`rho = 28`. Chaos is not automatic — it appears only in part of the parameter range.

In [ ]:
print(" rho   steady states   Lyapunov exponent   behaviour")
for rho in (0.5, 5.0, 15.0, 24.0, 28.0, 40.0):
    exponent_here = largest_lyapunov(rho=rho, renormalisations=800, discard=150)
    verdict = "chaotic" if exponent_here > 0.01 else "settles down"
    print(f"{rho:5.1f}   {len(fixed_points(rho=rho)):^13d}   {exponent_here:+17.3f}   {verdict}")

- At which `rho` does the exponent first turn positive? **TODO**
- Below `rho = 1` there is only one steady state. What happened to the other two? **TODO**
- A negative exponent means nearby states converge. What does the trajectory look like
  then — and would a forecast be useful? **TODO**

Re-run section 4 with a `rho` from the "settles down" rows and look at the shape. Chaos is
a property of a system *in a regime*, not a property of having three equations.

## 9. What this does and does not say

Write one or two sentences on each:

1. **This is not a weather model.** Three variables, no geography, no water. It shows that
   a deterministic system can be unpredictable; it forecasts nothing. **TODO**
2. **Unpredictable is not random.** Every run here is exactly reproducible from its initial
   state — rerun any cell and you get identical numbers. **TODO**
3. **Weather and climate are different questions.** The trajectory is unpredictable after a
   few tens of time units, yet the *attractor* — the set of states visited, and how often —
   is perfectly stable. Which of those two is a climate projection about? **TODO**
4. **Changing a parameter changes the attractor.** In section 8, `rho` changed the shape of
   the object itself. What is the climate analogue of turning up `rho`? **TODO**

Point 3 is the one worth carrying out of the room: chaos limits weather prediction without
making climate prediction impossible, because they are not the same claim.

> Reference: Lorenz, E. N. (1963), Deterministic Nonperiodic Flow, *Journal of the
> Atmospheric Sciences*, 20(2), 130-141.

**Exit ticket.** In two sentences: a colleague says "climate models can't be trusted
because weather is chaotic." Using the attractor, say what is right and what is wrong
about that.